In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import zipfile

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB

## 1. Import Libraries

This cell imports all libraries needed for the Phase 6 classification pipeline.  
We use:
- **pandas / numpy** for data handling and numeric operations
- **Path / zipfile** to load CSVs from disk or inside zipped folders
- **scikit-learn** tools for splitting data, preprocessing (encoding/scaling), training models, and evaluating performance

Keeping imports centralized supports reproducibility and makes it easy for teammates to run the notebook end-to-end.


In [3]:
def load_csv(path_or_zip, inner_csv=None):
    """
    Load a CSV from disk.
    If inner_csv is provided, treat path_or_zip as a .zip and read inner_csv from it.
    """
    path_or_zip = Path(path_or_zip)

    if inner_csv is None:
        return pd.read_csv(path_or_zip, low_memory=False)

    # read from zip
    with zipfile.ZipFile(path_or_zip) as z:
        with z.open(inner_csv) as f:
            return pd.read_csv(f, low_memory=False)

## 2. Generic Data Loader

This function standardizes data loading for Phase 6.  
It allows us to:
- load a normal CSV directly from a path, or
- load a CSV stored inside a `.zip` archive without manually extracting it.

The goal is to avoid hardcoding dataset-specific loading code so the same pipeline runs on multiple datasets.


In [4]:
def standardize_columns(df):
    df = df.copy()
    df.columns = (
        df.columns.str.strip()
                  .str.upper()
                  .str.replace(" ", "_")
    )
    return df


def basic_cleanup(df, missing_thresh=0.30, uniq_thresh=0.95,
                  protected=None, name="dataset", verbose=True):
    protected = set(protected or [])
    n_rows = len(df)

    # drop >30% missing
    missing_frac = df.isna().mean()
    drop_missing = [c for c, frac in missing_frac.items()
                    if frac > missing_thresh and c not in protected]
    df = df.drop(columns=drop_missing)

    # drop ID-like cols
    drop_ids = []
    for c in df.columns:
        if c in protected:
            continue
        ratio = df[c].nunique(dropna=True) / n_rows
        if ratio >= uniq_thresh:
            drop_ids.append(c)

    df = df.drop(columns=drop_ids)

    if verbose:
        print(f"\n=== Clean-up report for {name} ===")
        print(f"Dropped (> {missing_thresh*100:.0f}% missing): {drop_missing}")
        print(f"Dropped ID-like (>= {uniq_thresh*100:.0f}% unique): {drop_ids}")
        print(f"Final shape: {df.shape}")

    return df


def drop_low_variance(df, thresh=0.99, protected=None, name="dataset", verbose=True):
    protected = set(protected or [])
    drop_cols = []

    for c in df.columns:
        if c in protected:
            continue
        top_frac = df[c].value_counts(normalize=True, dropna=False).iloc[0]
        if top_frac >= thresh:
            drop_cols.append(c)

    df2 = df.drop(columns=drop_cols)

    if verbose:
        print(f"\n=== Low-variance drop for {name} ===")
        print(f"Dropped (>= {thresh*100:.0f}% same value): {drop_cols}")
        print(f"Shape after low-variance drop: {df2.shape}")

    return df2, drop_cols

## 3. Generalized Clean-Up Functions (Phase 6 Requirement)

These are the refactored Phase 4 clean-up steps, generalized for any wildfire dataset:

1. **standardize_columns()**  
   Makes column names consistent across datasets to prevent later mismatches (uppercase, no spaces).

2. **basic_cleanup()**  
   Performs the two required cleaning rules:
   - drops columns missing more than 30% of values
   - removes ID-like columns (≥95% unique values), since they add no predictive value.

3. **drop_low_variance()**  
   Removes near-constant columns where one value dominates ≥99% of entries.  
   These columns typically carry no useful signal for classification.

Protected columns ensure that target variables (ex: `LARGEFIRE`, `FIRE_SIZE`) are never dropped.


In [10]:
print("NM columns:")
print(nm_df.columns.tolist())

NM columns:
['SOURCE_SYSTEM_TYPE', 'SOURCE_SYSTEM', 'NWCG_REPORTING_AGENCY', 'NWCG_REPORTING_UNIT_ID', 'NWCG_REPORTING_UNIT_NAME', 'SOURCE_REPORTING_UNIT', 'SOURCE_REPORTING_UNIT_NAME', 'FIRE_NAME', 'FIRE_YEAR', 'DISCOVERY_DATE', 'DISCOVERY_DOY', 'DISCOVERY_TIME', 'STAT_CAUSE_CODE', 'STAT_CAUSE_DESCR', 'CONT_DATE', 'CONT_DOY', 'CONT_TIME', 'FIRE_SIZE', 'FIRE_SIZE_CLASS', 'LATITUDE', 'LONGITUDE', 'OWNER_CODE', 'OWNER_DESCR', 'SHAPE']


In [11]:
def ensure_target(df, target_col="LARGEFIRE", size_col_candidates=None, threshold=100.0):
    """
    Ensure binary target exists.
    - If target_col already exists, use it.
    - Else, try to build from the first matching size column in candidates.
    """
    df = df.copy()
    cols = set(df.columns)

    # If target already exists, we're done
    if target_col in cols:
        df[target_col] = df[target_col].astype(int)
        return df, target_col

    # Candidate size columns to try
    if size_col_candidates is None:
        size_col_candidates = [
            "FIRE_SIZE", "FIRE_SIZE_ACRES", "SIZE", "ACRES",
            "TOTAL_ACRES", "FINAL_ACRES", "BURNED_AREA"
        ]

    # Find first candidate that exists
    for sc in size_col_candidates:
        if sc in cols:
            df[target_col] = (pd.to_numeric(df[sc], errors="coerce") >= threshold).astype(int)
            return df, target_col

    raise ValueError(
        f"Could not find target '{target_col}' or any size column in {size_col_candidates}."
    )

## 4. Ensure a Binary Target for Classification

Different datasets label “fire size” differently.  
This function guarantees we always end with the same target:

- If `LARGEFIRE` already exists, it is reused.
- If not, the function searches for any reasonable fire-size column
  (using common names or fuzzy matching) and builds a binary label using a threshold (default 100 acres).

This keeps modeling consistent for both USFS and NM datasets, even if they store size under different column names.


In [18]:
# Cell 12 — build/ensure targets for BOTH datasets

# USFS: build LargeFire from FIRE_SIZE (or keep if already exists)
usfs_df, _ = ensure_target(usfs_df, target_col="LARGEFIRE", threshold=100)

# NM: your NM columns DO include FIRE_SIZE, so this will work now
nm_df, _ = ensure_target(nm_df, target_col="LARGEFIRE", threshold=100)

print("USFS final shape:", usfs_df.shape)
print("NM final shape:", nm_df.shape)
print("NM LargeFire positive rate:", nm_df["LARGEFIRE"].mean())

USFS final shape: (582291, 20)
NM final shape: (37478, 25)
NM LargeFire positive rate: 0.06313036981695928


## Build a Shared Classification Target on Both Datasets

This cell creates a consistent binary target variable called **`LARGEFIRE`** for both datasets.  
Phase 6 requires our pipeline to work across different datasets, so we need a single target name and definition.

- For **USFS**, the target is built from a fire-size field if `LARGEFIRE` doesn’t already exist.
- For **NM_Wildfires**, the same logic is applied, ensuring that both datasets can feed into the same models.

We also print dataset shapes and the NM positive rate so we can confirm the target was successfully created and understand class imbalance before modeling.

In [13]:
print("USFS columns:")
print(usfs_df.columns.tolist())

print("\nNM columns:")
print(nm_df.columns.tolist())

USFS columns:
['X', 'Y', 'GLOBALID', 'CN', 'FIRENAME', 'FIREYEAR', 'UNIQFIREID', 'SOFIRENUM', 'SIZECLASS', 'TOTALACRES', 'STATCAUSE', 'DATASOURCE', 'OWNERAGENCY', 'PROTECTIONAGENCY', 'POINTTYPE', 'PERIMEXISTS', 'FIRERPTQC', 'DBSOURCEID', 'DBSOURCEDATE']

NM columns:
['SOURCE_SYSTEM_TYPE', 'SOURCE_SYSTEM', 'NWCG_REPORTING_AGENCY', 'NWCG_REPORTING_UNIT_ID', 'NWCG_REPORTING_UNIT_NAME', 'SOURCE_REPORTING_UNIT', 'SOURCE_REPORTING_UNIT_NAME', 'FIRE_NAME', 'FIRE_YEAR', 'DISCOVERY_DATE', 'DISCOVERY_DOY', 'DISCOVERY_TIME', 'STAT_CAUSE_CODE', 'STAT_CAUSE_DESCR', 'CONT_DATE', 'CONT_DOY', 'CONT_TIME', 'FIRE_SIZE', 'FIRE_SIZE_CLASS', 'LATITUDE', 'LONGITUDE', 'OWNER_CODE', 'OWNER_DESCR', 'SHAPE']


## Inspect Column Names to Debug Target Creation

This block prints the column names for both datasets after cleanup and standardization.  
We used this step to confirm which fire-size or target columns still exist, since column names can change between datasets.

This inspection is important for Phase 6 because the pipeline must adapt to different schemas instead of relying on hard-coded names.

In [14]:
def ensure_target(df, target_col="LARGEFIRE", threshold=100.0):
    """
    Ensure binary target exists.
    - If target_col exists, use it.
    - Else, find a fire-size-like column by name pattern and build target from it.
    """
    df = df.copy()
    cols = list(df.columns)
    cols_set = set(cols)

    # If target already exists, use it
    if target_col in cols_set:
        df[target_col] = df[target_col].astype(int)
        return df, target_col

    # Try exact common names first
    common_candidates = [
        "FIRE_SIZE", "FIRE_SIZE_ACRES", "FINAL_FIRE_SIZE", "SIZE", "ACRES",
        "TOTAL_ACRES", "FINAL_ACRES", "BURNED_AREA"
    ]
    for sc in common_candidates:
        if sc in cols_set:
            df[target_col] = (pd.to_numeric(df[sc], errors="coerce") >= threshold).astype(int)
            return df, target_col

    # Fuzzy match: look for any column containing FIRE and SIZE or ACRE
    fuzzy_candidates = [
        c for c in cols
        if ("FIRE" in c and "SIZE" in c) or ("ACRE" in c)
    ]
    if fuzzy_candidates:
        sc = fuzzy_candidates[0]  # take the first reasonable match
        df[target_col] = (pd.to_numeric(df[sc], errors="coerce") >= threshold).astype(int)
        print(f"[ensure_target] Built {target_col} from fuzzy size column: {sc}")
        return df, target_col

    raise ValueError(
        f"Could not find target '{target_col}' or any fire-size-like column to build it."
    )

## Generalized Target Builder (`ensure_target`)

Different wildfire datasets store fire size under different names.  
This function guarantees that a **binary target (`LARGEFIRE`) always exists**:

1. If `LARGEFIRE` already exists, we keep it.
2. Otherwise, we search for common size fields (ex: `FIRE_SIZE`, `TOTALACRES`).
3. If no exact match exists, we use fuzzy matching to find any column containing patterns like:
   - `"FIRE" + "SIZE"`
   - `"ACRE"`

Once a size column is found, fires ≥ 100 acres are labeled as large fires (`1`), and smaller fires as (`0`).

This lets the same classifiers run cleanly on both USFS and NM data without manual renaming.

In [15]:
usfs_df, _ = ensure_target(usfs_df, target_col="LARGEFIRE", threshold=100)
nm_df, _   = ensure_target(nm_df, target_col="LARGEFIRE", threshold=100)

print("USFS final shape:", usfs_df.shape)
print("NM final shape:", nm_df.shape)
print("USFS LargeFire rate:", usfs_df["LARGEFIRE"].mean())
print("NM LargeFire rate:", nm_df["LARGEFIRE"].mean())


[ensure_target] Built LARGEFIRE from fuzzy size column: TOTALACRES
USFS final shape: (582291, 20)
NM final shape: (37478, 25)
USFS LargeFire rate: 0.027520603959188792
NM LargeFire rate: 0.06313036981695928


## Verify Targets Were Built Correctly

This block confirms that the generalized target builder worked:

- USFS successfully generated `LARGEFIRE` using **`TOTALACRES`** as the size column.
- NM used its existing fire size field.
- Final dataset shapes were printed so we know how many features remain after cleanup.
- Positive class rates were printed to document class imbalance before model training.

These checks are required for reproducibility and for interpreting model performance later.

In [6]:
def make_preprocessor(X, max_cat_levels=100):
    """
    Build ColumnTransformer:
    - numeric -> StandardScaler
    - categorical -> OneHotEncoder (limited)
    """
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = [c for c in X.columns if c not in num_cols]

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), num_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore",
                                  max_categories=max_cat_levels,
                                  sparse_output=False), cat_cols)
        ],
        remainder="drop"
    )
    return preprocessor, num_cols, cat_cols

## Build a Reusable Preprocessor for Mixed Feature Types

Wildfire datasets contain both numeric and categorical variables.  
This function creates a **ColumnTransformer** that applies:

- **StandardScaler** to numeric columns (so models like LR/KNN aren’t biased by scale)
- **OneHotEncoder** to categorical columns (so models can use non-numeric predictors)
- `max_cat_levels=100` prevents feature explosion when categories are very large

This matches the Phase 4 approach, but generalized so it works for multiple datasets in Phase 6.

In [7]:
def get_models(task_set="USFS"):
    """
    task_set="USFS" -> LR, RF, KNN
    task_set="ALT"  -> LR, DT, NB
    """
    if task_set == "USFS":
        return {
            "LogisticRegression": LogisticRegression(
                max_iter=2000, class_weight="balanced", random_state=42
            ),
            "RandomForest": RandomForestClassifier(
                n_estimators=300, class_weight="balanced_subsample", random_state=42, n_jobs=-1
            ),
            "KNN": KNeighborsClassifier(n_neighbors=15)
        }
    else:
        return {
            "LogisticRegression": LogisticRegression(
                max_iter=2000, class_weight="balanced", random_state=42
            ),
            "DecisionTree": DecisionTreeClassifier(
                class_weight="balanced", random_state=42
            ),
            "NaiveBayes": GaussianNB()
        }

## 6. Define Phase 4 Model Families

Phase 4 used different model sets for different datasets.  
This function bundles those into reusable dictionaries:

- **USFS set:** Logistic Regression, Random Forest, KNN  
- **ALT set:** Logistic Regression, Decision Tree, Naive Bayes  

By generalizing the model definitions, we avoid rewriting model code for each dataset in Phase 6.

In [8]:
def train_and_evaluate(df, target_col="LARGEFIRE", test_size=0.30, seed=42,
                       task_set="USFS"):
    """
    Full classification run:
    - split
    - preprocess with pipeline
    - CV F1 on train
    - test accuracy/precision/recall/F1
    """

    # Separate X/y
    X = df.drop(columns=[target_col])
    y = df[target_col].astype(int)

    # Train/test split (stratified)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=seed
    )

    # Preprocessor
    preprocessor, num_cols, cat_cols = make_preprocessor(X_train)

    models = get_models(task_set=task_set)

    results = []

    for name, clf in models.items():
        pipe = Pipeline([
            ("prep", preprocessor),
            ("clf", clf)
        ])

        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
        cv_f1 = cross_val_score(pipe, X_train, y_train, cv=cv,
                                scoring="f1", n_jobs=-1)
        cv_mean, cv_std = cv_f1.mean(), cv_f1.std()

        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_test)

        res = {
            "Model": name,
            "CV_F1_mean": cv_mean,
            "CV_F1_std": cv_std,
            "Test_Accuracy": accuracy_score(y_test, y_pred),
            "Test_Precision": precision_score(y_test, y_pred, zero_division=0),
            "Test_Recall": recall_score(y_test, y_pred, zero_division=0),
            "Test_F1": f1_score(y_test, y_pred, zero_division=0),
            "ConfusionMatrix": confusion_matrix(y_test, y_pred),
            "Report": classification_report(y_test, y_pred, zero_division=0)
        }
        results.append(res)

        print(f"\n{name} CV F1: {cv_mean:.3f} +/- {cv_std:.3f}")
        print(res["Report"])

    return pd.DataFrame(results), (X_train, X_test, y_train, y_test)

## Generalized Training + Evaluation Pipeline

This function wraps the full Phase 4 classification workflow into one reusable unit:

### What it does:
1. Splits the dataset into **stratified train/test sets**  
   (important because large fires are rare).
2. Applies preprocessing through a pipeline.
3. Trains the selected model family (USFS set or ALT set).
4. Runs **5-fold cross-validation using F1 score** on the training set  
   (F1 is more meaningful than accuracy under class imbalance).
5. Evaluates on the test set using:
   - accuracy
   - precision
   - recall
   - F1
   - confusion matrix
   - classification report

### Why we do this:
Phase 6 requires a pipeline that can be reused for different datasets without rewriting model code.  
This function ensures consistent evaluation and easy comparison across datasets and model types.


In [17]:
phase6_dir = Path(r"C:\Users\clawr\OneDrive\Desktop\Fall 2025\CSCI 5415 (Data Mining)\Group Project\Phase 6")

# USFS normal CSV
usfs_path = phase6_dir / "cleaned_National_USFS .csv"
usfs_df = load_csv(usfs_path)

# NM Wildfires inside zip
datasets_zip = phase6_dir / "Datasets.zip"
nm_df = load_csv(datasets_zip, inner_csv="Datasets/NM_Wildfires.csv")

# CLEANUP
protected = ["LARGEFIRE", "FIRE_SIZE"]

usfs_df = drop_low_variance(
    basic_cleanup(standardize_columns(usfs_df), protected=protected, name="USFS"),
    protected=protected, name="USFS"
)[0]

nm_df = drop_low_variance(
    basic_cleanup(standardize_columns(nm_df), protected=protected, name="NM"),
    protected=protected, name="NM"
)[0]

# TARGET (fixed)
usfs_df, _ = ensure_target(usfs_df, target_col="LARGEFIRE", threshold=100)
nm_df, _   = ensure_target(nm_df, target_col="LARGEFIRE", threshold=100)

print("USFS final shape:", usfs_df.shape)
print("NM final shape:", nm_df.shape)
print("USFS LargeFire rate:", usfs_df["LARGEFIRE"].mean())
print("NM LargeFire rate:", nm_df["LARGEFIRE"].mean())



=== Clean-up report for USFS ===
Dropped (> 30% missing): []
Dropped ID-like (>= 95% unique): ['OBJECTID']
Final shape: (582291, 20)

=== Low-variance drop for USFS ===
Dropped (>= 99% same value): ['FIRETYPECATEGORY']
Shape after low-variance drop: (582291, 19)

=== Clean-up report for NM ===
Dropped (> 30% missing): ['LOCAL_FIRE_REPORT_ID', 'LOCAL_INCIDENT_ID', 'FIRE_CODE', 'ICS_209_INCIDENT_NUMBER', 'ICS_209_NAME', 'MTBS_ID', 'MTBS_FIRE_NAME', 'COMPLEX_NAME', 'COUNTY', 'FIPS_CODE', 'FIPS_NAME']
Dropped ID-like (>= 95% unique): ['OBJECTID', 'FOD_ID', 'FPA_ID']
Final shape: (37478, 25)

=== Low-variance drop for NM ===
Dropped (>= 99% same value): ['STATE']
Shape after low-variance drop: (37478, 24)
[ensure_target] Built LARGEFIRE from fuzzy size column: TOTALACRES
USFS final shape: (582291, 20)
NM final shape: (37478, 25)
USFS LargeFire rate: 0.027520603959188792
NM LargeFire rate: 0.06313036981695928


## 8. Apply the Generalized Pipeline to Both Datasets

This block executes the generalized preprocessing on:

- **USFS cleaned dataset**
- **NM_Wildfires dataset (loaded from zip)**

Steps applied:
1. load data sources in a reproducible way
2. standardize column names
3. remove high-missing and ID-like columns
4. remove low-variance columns
5. ensure a shared binary target (`LARGEFIRE`)

Printing final shapes and class rates provides documentation for the Phase 6 report and confirms the pipeline works on both domains before modeling.
